# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [2]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。


In [3]:
# 创建三个浮点标量参数 w1、w2、b，都启用梯度
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# 三个参数一起交给同一个优化器
optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

# 训练前先记录初始状态，方便之后判断训练有没有效果
with torch.no_grad():
    init_pred = predict(x1, x2, w1, w2, b)
    init_loss = torch.abs(y - init_pred).mean()
print('初始参数：w1 = 0.0, w2 = 0.0, b = 0.0')
print('初始 y_pred:', init_pred, ' shape:', tuple(init_pred.shape))
print('初始平均损失:', round(init_loss.item(), 6), ' shape:', tuple(init_loss.shape))
print()

for epoch in range(1, 2001):
    # 设置本轮学习率，并清空上一轮留下的旧梯度
    optimizer.param_groups[0]['lr'] = 0.1 / (epoch ** 0.5)
    optimizer.zero_grad()

    # 调用给定的 predict，同时传入 x1 和 x2，得到 shape 为 (6,) 的 y_pred
    y_pred = predict(x1, x2, w1, w2, b)

    # 先得到 (6,) 的逐样本绝对误差，再用 .mean() 得到 shape 为 () 的平均损失
    per_sample_loss = torch.abs(y - y_pred)
    loss = per_sample_loss.mean()

    # 只对这个平均损失 backward 一次，然后更新全部参数
    loss.backward()
    optimizer.step()

# 在 torch.no_grad() 中，用训练后的参数重新计算预测与损失
with torch.no_grad():
    final_pred = predict(x1, x2, w1, w2, b)
    final_per_sample = torch.abs(y - final_pred)
    final_loss = final_per_sample.mean()

print('最终参数：w1 =', round(w1.item(), 4),
      ', w2 =', round(w2.item(), 4),
      ', b =', round(b.item(), 4))
print()
print('x1:', x1, ' shape:', tuple(x1.shape))
print('x2:', x2, ' shape:', tuple(x2.shape))
print('y: ', y, ' shape:', tuple(y.shape))
print()
print('y_pred:', final_pred)
print('y_pred shape:', tuple(final_pred.shape))
print('逐样本损失:', final_per_sample)
print('逐样本损失 shape:', tuple(final_per_sample.shape))
print('平均损失:', round(final_loss.item(), 6), ' shape:', tuple(final_loss.shape))


初始参数：w1 = 0.0, w2 = 0.0, b = 0.0
初始 y_pred: tensor([0., 0., 0., 0., 0., 0.])  shape: (6,)
初始平均损失: 4.333333  shape: ()

最终参数：w1 = 1.999 , w2 = -3.001 , b = 0.9994

x1: tensor([-2., -1.,  0.,  1.,  2.,  3.])  shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.])  shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.])  shape: (6,)

y_pred: tensor([-5.9997, -0.9996, -5.0026,  5.9994, -4.0056,  3.9954])
y_pred shape: (6,)
逐样本损失: tensor([0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046])
逐样本损失 shape: (6,)
平均损失: 0.002373  shape: ()


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）

一次预测需要 **2 个输入张量**：x1 和 x2，一起传给 `predict(x1, x2, w1, w2, b)`。

它们的 shape 都是 `(6,)`，各有 6 个元素，对应 6 个样本。关键是 `x1[i]`、`x2[i]`、`y[i]` 属于**同一个样本**：`x1[i]` 和 `x2[i]` 是这个样本的两个输入，`y[i]` 是它的目标。所以样本数 6 是"竖着"数有几行，而 x1、x2 是"横着"的两个输入通道，两者不是一回事。

一次 `predict` 调用同时吃进 x1、x2 两个 `(6,)` 张量，返回一个 `(6,)` 的 y_pred，第 i 个预测就是用第 i 个样本的两个输入算出来的：`y_pred[i] = w1*x1[i] + w2*x2[i] + b`。输出 shape `(6,)` 正好等于样本数，说明 6 个样本是一次批量算完的。

（对比作业 01：那里一个样本只有 2 个输入值，装成 shape `(2,)`；这里 6 个样本、每个 2 个输入，分别装进 x1、x2 两个 `(6,)` 张量。别把"2 个输入"和"6 个样本"搞混——2 是特征数，6 是样本数。）

2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）

- **预测公式**：上周作业 01 是 `x * w**2`（单输入、参数被平方），这周改成 `predict` 里的 `w1*x1 + w2*x2 + b`（两个输入、对参数是线性的）。
- **参数列表**：上周只有一个 w，这周是 w1、w2、b 三个，且 w1 配 x1、w2 配 x2，各自管一个输入通道，再加一个偏置 b。三个一起放进 `SGD([w1, w2, b], lr=...)`。
- **损失计算**：仍然是绝对值损失 `torch.abs(y - y_pred).mean()`，和上周一样没有换成平方误差；只是因为现在有 6 个样本、`predict` 要传两个输入，y_pred 的算法变了。
- **优化器的基本步骤没变**：还是"设学习率 → `zero_grad()` 清空旧梯度 → 前向预测 → 算 loss → `loss.backward()` 反向 → `optimizer.step()` 更新"这一套。变的只是喂给 `predict` 的输入从一个 x 变成 x1、x2 两个，参数从一个变成三个。学习率沿用题目建议的 `0.1 / (epoch**0.5)`，跑 2000 轮。

3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）

- 最终参数：w1 ≈ **1.999**，w2 ≈ **-3.001**，b ≈ **0.9994**（非常接近整数 2、-3、1）。
- 最终 y_pred ≈ `[-5.9997, -0.9996, -5.0026, 5.9994, -4.0056, 3.9954]`，真实 y = `[-6, -1, -5, 6, -4, 4]`，两者几乎逐个对上。
- 逐样本损失 ≈ `[0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046]`，平均损失 ≈ **0.002373**。

怎么判断训练有效：

- **看损失降了多少**：初始 w1=w2=b=0 时，y_pred 全是 0，平均损失是 4.333333；训练后降到 0.002373，下降了三个数量级。
- **看参数是否收敛到合理值**：学到的 (1.999, -3.001, 0.9994) 很接近 (2, -3, 1)。代回公式 y = 2*x1 - 3*x2 + 1 验证：样本 0 是 2*(-2) - 3*1 + 1 = -6 = y[0]；样本 3 是 2*1 - 3*(-1) + 1 = 6 = y[3]。六个样本都能对上，说明数据本来就是按 y = 2*x1 - 3*x2 + 1 生成的，训练把它学出来了。
- **看每个样本是否都拟合，而不只是平均值低**：6 个逐样本损失都在 0.006 以内，没有哪个样本被落下。

综合这三点，判断训练是有效的。

4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

因为这次的预测公式对参数是**线性**的：`y_pred = w1*x1 + w2*x2 + b`，参数以一次方出现。求导后 d(y_pred)/d(w1) = x1、d(y_pred)/d(w2) = x2、d(y_pred)/d(b) = 1，这些梯度只跟输入 x1、x2 有关，**跟参数自己当前的值无关**。所以哪怕 w1=w2=b=0，只要输入不全为零，梯度就不是零。实测零初始化时 w1.grad ≈ -0.833、w2.grad = 1.0、b.grad ≈ 0.333，都非零，参数第一步就能动起来。

这和上周"参数被平方"的例子正好相反。上周作业 01 里预测是 `x * w**2`，梯度 d/dw = 2*x*w，里面带了个 w；一旦 w=0，梯度就是 2*x*0 = 0，参数会卡在零点出不来（实测 w=0 时 w.grad = 0，要先把 w 设成 0.5 才有梯度 -1.5）。所以那种写法不能从零初始化，而这次的线性公式可以从零开始——这就是题目说"这里预测公式是线性的，零初始化与上周的平方参数例子不同"的原因。